## Comparing dataset projections (Part 2)

In [69]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Dynamic World (re-projection standard)

In [82]:
START = "2024-01-01"
END = "2024-12-31"

# 10m resolution

dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

dw_mode = dw_col.select("label").mode()

# reprojecting mode back to original Dynamic World projection
sample_image = dw_col.first()
collection_projection = sample_image.projection()

reprojected_mode = (
    dw_mode.resample("bilinear")
    .reproject(collection_projection)
    .rename("mode_reprojected")
    .clip(panama_geom)
)

VIS_PALETTE = [
    "419bdf", "397d49", "88b053", "7a87c6",
    "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"
]

Map.addLayer(
    reprojected_mode,
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World"
)

In [84]:
reprojected_mode.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:32617',
 'transform': [10, 0, 267490, 0, 10, 790400]}

### Slope

In [76]:
# ~30 m resolution
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee").clip(panama_geom)

# reprojecting to fit other data layers
sample_image = dw_mode
collection_projection = sample_image.projection()

reprojected_slope = (
    slope.resample("bilinear")
    .reproject(collection_projection)
    .rename("slope_reprojected")
    .clip(panama_geom)
)

Map.addLayer(slope, {"min": 0, "max": 30}, "Slope")
Map.addLayer(reprojected_slope, {"min": 0, "max": 30}, "Reprojected Slope")

Map 

Map(bottom=16050803.0, center=[7.772832423028278, -80.45374989509584], controls=(WidgetControl(options=['posit…

In [78]:
reprojected_slope.projection().nominalScale().getInfo()

111319.49079327357

In [79]:
slope.projection().nominalScale().getInfo()

30.974316829122042

### Soils

In [56]:
# ~260m resolution
pa = ee.Image("projects/deforestation-495419/assets/Soil_Org_C").clip(panama_geom)
pn = ee.Image("projects/deforestation-495419/assets/Soil_N").clip(panama_geom)
ps = ee.Image("projects/deforestation-495419/assets/sand").clip(panama_geom)
ph = ee.Image("projects/deforestation-495419/assets/pH").clip(panama_geom)

Map.addLayer(pa, {"min": 0, "max": 150}, "Soil Organic Carbon")
Map.addLayer(pn, {"min": 0, "max": 80}, "Nitrogen")
Map.addLayer(ps, {"min": 0, "max": 1000}, "Sand")
Map.addLayer(ph, {"min": 40, "max": 80}, "pH")

In [57]:
pa.projection().getInfo()
pn.projection().getInfo()
ps.projection().getInfo()
ph.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.0022607385079125844, 0, -83, 0, -0.002390438247011952, 10]}

### Distance to deforested areas

In [ ]:
# ~28m resolution
hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(panama_geom)

lossyear = hansen.select("lossyear")

recent_loss = lossyear.gte(20).selfMask()

distance_loss = (
    recent_loss.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .rename("dist_loss")
    .clip(panama_geom)
)

# Map.addLayer(recent_loss, {"palette": ["red"]}, "Recent Loss")
Map.addLayer(distance_loss, {"min": 0, "max": 5000}, "Distance to Loss")

In [ ]:
distance_loss.projection().getInfo()

27.829872698318393

### Precipitation (monthly)

In [60]:
# ~5566m resolution
chirps = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_SAT")
    .filterDate("2018-05-01", "2018-05-31")
    .select("precipitation")
)

# precip = chirps.mean().clip(panama_geom) --> lowers resolution BIG TIME (~111,319m)

Map.addLayer(
    chirps,
    {
        "min": 1,
        "max": 17,
        "palette": ["#001137", "#0aab1e", "#e7eb05", "#2c7fb8", "#253494"]
    },
    "Precipitation"
)

In [61]:
chirps.first().projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.05000000074505806, 0, -180, 0, -0.05000000074505806, 60]}

### Temperature (monthly)

In [62]:
# ~11,131m resolution
temp = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
    .filterDate("2023-01-01", "2023-02-01")
    .first()
    .clip(panama_geom)
)

Map.addLayer(
    temp,
    {
        "bands": ["temperature_2m"],
        "min": 250,
        "max": 320
    },
    "Temperature"
)

In [63]:
temp.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.1, 0, -180.05, 0, -0.1, 90.05]}

In [64]:
temp.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:4326',
 'transform': [0.1, 0, -180.05, 0, -0.1, 90.05]}

### Distance to hydrological basins

In [65]:
# 100m resolution
basins = ee.Image("projects/deforestation-495419/assets/hydrographic_basins").clip(panama_geom)

basins_int = basins.toInt()
neighbors = basins_int.focal_mode(1)

edges = basins_int.neq(neighbors).selfMask()

dist_basin = (
    edges.fastDistanceTransform(512).sqrt()
    .multiply(basins.projection().nominalScale())
    .clip(panama_geom)
)

# Map.addLayer(basins, {}, "Basins")
# Map.addLayer(edges, {"palette": ["red"]}, "Basin Edges")
Map.addLayer(dist_basin, {"min": 0, "max": 50000}, "Distance Basin")

In [66]:
dist_basin.projection().getInfo()

{'type': 'Projection',
 'crs': 'EPSG:3857',
 'transform': [101.30653541413305,
  0,
  -9246721.428728944,
  0,
  -101.3065354141332,
  1168336.6903672446]}

### Generate map

In [67]:
Map

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…